## Harry Potter RAG-Powered Assistant

This project implements a Retrieval-Augmented Generation (RAG) Assistant that answers natural language questions based on the seven Harry Potter books

### Number of documents: 1 document
### File name: harrypotter.pdf
### Format: PDF
### Number of pages:3623

## 1. Environment Setup

install and import the required libraries and load the project configuration from the .env file

The .env file contains the API keys, model names, Qdrant URL, and collection name used by the RAG pipeline

In [71]:
!pip install -q pymupdf sentence-transformers qdrant-client python-dotenv
!pip install -q langchain-groq langchain-google-genai

In [72]:
import os
import re
import torch
from pathlib import Path

import pymupdf

from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient, models

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

In [73]:
load_dotenv("/content/.env")

QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION")
TOP_K = int(os.getenv("TOP_K", "3"))

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [74]:
print("API keys loaded:",
      all([
          QDRANT_API_KEY,
          GEMINI_API_KEY,
          GROQ_API_KEY
      ]))

API keys loaded: True


## 2. Document Preparation

The source data consists of the seven Harry Potter books combined into a single PDF file

First, we convert the PDF into a Markdown file

Each PDF page is stored in the Markdown file with a page heading so that we can later identify and separate the pages.

In [75]:
print("Start RAG")

def pdf_to_markdown(pdf_path, markdown_path):
    """Extract the text from a PDF and save it as a Markdown file."""

    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    print(f"Markdown file created: {markdown_path}")

Start RAG


In [76]:
pdf_path = "/content/harrypotter.pdf"
markdown_path = "/content/output.md"

pdf_to_markdown(pdf_path, markdown_path)

Markdown file created: /content/output.md


In [77]:
output_file = Path("/content/output.md")

print("File exists:", output_file.exists())

File exists: True


In [78]:
print(output_file.read_text(encoding="utf-8")[:2000])

## Page 1



## Page 2



## Page 3



## Page 4



## Page 5



## Page 6

CONTENTS
Harry Potter and the Sorcerer’s Stone
Harry Potter and the Chamber of Secrets
Harry Potter and the Prisoner of Azkaban
Harry Potter and the Goblet of Fire
Harry Potter and the Order of the Phoenix
Harry Potter and the Half-Blood Prince
Harry Potter and the Deathly Hallows


## Page 7



## Page 8



## Page 9

 
FOR JESSICA, WHO LOVES STORIES,
FOR ANNE, WHO LOVED THEM TOO;
AND FOR DI, WHO HEARD THIS ONE FIRST.


## Page 10

 
CONTENTS
ONE
The Boy Who Lived
TWO
The Vanishing Glass
THREE
The Letters from No One
FOUR
The Keeper of the Keys
FIVE
Diagon Alley
SIX
The Journey from Platform Nine and Three-quarters
SEVEN
The Sorting Hat
EIGHT
The Potions Master
NINE
The Midnight Duel
TEN
Halloween
ELEVEN
Quidditch
TWELVE


## Page 11

The Mirror of Erised
THIRTEEN
Nicolas Flamel
FOURTEEN
Norbert the Norwegian Ridgeback
FIFTEEN
The Forbidden Forest
SIXTEEN
Through the Trapdoor
SEVENTEEN
The Man with Two Faces



## 3. Chunking

The extracted Markdown file contains the complete PDF

the document will be split into page-level chunks

Each page will be stored as a separate Markdown file

Each page chunk will contain:

- Book name
- Page number
- Page content

The book boundaries are defined using the page ranges of the seven books

In [79]:
INPUT_FILE = Path("/content/output.md")

OUTPUT_FOLDER = Path("/content/dataset")

BOOK_RANGES = [
    ("Harry Potter and the Sorcerer Stone", 12, 274),
    ("Harry Potter and the Chamber of Secrets", 282, 565),
    ("Harry Potter and the Prisoner of Azkaban", 573, 939),
    ("Harry Potter and the Goblet of Fire", 949, 1560),
    ("Harry Potter and the Order of the Phoenix", 1570, 2406),
    ("Harry Potter and the Half-Blood Prince", 2409, 2964),
    ("Harry Potter and the Deathly Hallows", 2974, 3622),
]

In [80]:
def get_book_name(page_number):
    """Return the book name for a given page number"""

    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name

    return None

In [81]:
def clean_text(text):
    """Clean the text by removing extra whitespace and unwanted characters"""

    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

In [82]:
def split_pages():
    """Split the Markdown file into separate files for each page"""

    text = INPUT_FILE.read_text(encoding="utf-8")

    pages = re.split(
        r"^##\s*Page\s+(\d+)\s*$",
        text,
        flags=re.MULTILINE
    )

    OUTPUT_FOLDER.mkdir(exist_ok=True)

    for i in range(1, len(pages), 2):

        page_number = int(pages[i])

        page_content = clean_text(pages[i + 1])

        book_name = get_book_name(page_number)

        if book_name:

            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )

            file_name = f"{book_name} - Page {page_number}.md"

            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")


In [83]:
split_pages()

In [84]:
files = sorted(OUTPUT_FOLDER.glob("*.md"))

print("Number of page chunks:", len(files))

Number of page chunks: 3568


### Inspect One Chunk to check

In [85]:
print(files[0].read_text(encoding="utf-8")[:2000])

# Harry Potter and the Chamber of Secrets

## Page 282

N CHAPTER ONE THE WORST BIRTHDAY ot for the first time, an argument had broken out over breakfast at number four, Privet Drive. Mr. Vernon Dursley had been woken in the early hours of the morning by a loud, hooting noise from his nephew Harry’s room. “Third time this week!” he roared across the table. “If you can’t control that owl, it’ll have to go!” Harry tried, yet again, to explain. “She’s bored,” he said. “She’s used to flying around outside. If I could just let her out at night —” “Do I look stupid?” snarled Uncle Vernon, a bit of fried egg dangling from his bushy mustache. “I know what’ll happen if that owl’s let out.” He exchanged dark looks with his wife, Petunia. Harry tried to argue back but his words were drowned by a long, loud belch from the Dursleys’ son, Dudley. “I want more bacon.” “There’s more in the frying pan, sweetums,” said Aunt Petunia, turning misty eyes on her massive son. “We must build you up while we’v

## 4. Embeddings

now will create an embedding for the content of every page

Each page will be represented as a numerical vector by using

intfloat/multilingual-e5-large

For each page, will keep three pieces of information as its payload:

- Book name
- Page number
- Page content

In [86]:
DATASET_FOLDER = Path("/content/dataset")

MODEL_NAME = "intfloat/multilingual-e5-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

def read_page(file_path):
    lines = file_path.read_text(encoding="utf-8").splitlines()

    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()

    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }

Device: cuda


In [87]:
files = sorted(DATASET_FOLDER.glob("*.md"))

pages = [read_page(file) for file in files]

print("Number of pages:", len(pages))

Number of pages: 3568


In [88]:
pages[0]

{'book_name': 'Harry Potter and the Chamber of Secrets',
 'page_number': 282,
 'content': 'N CHAPTER ONE THE WORST BIRTHDAY ot for the first time, an argument had broken out over breakfast at number four, Privet Drive. Mr. Vernon Dursley had been woken in the early hours of the morning by a loud, hooting noise from his nephew Harry’s room. “Third time this week!” he roared across the table. “If you can’t control that owl, it’ll have to go!” Harry tried, yet again, to explain. “She’s bored,” he said. “She’s used to flying around outside. If I could just let her out at night —” “Do I look stupid?” snarled Uncle Vernon, a bit of fried egg dangling from his bushy mustache. “I know what’ll happen if that owl’s let out.” He exchanged dark looks with his wife, Petunia. Harry tried to argue back but his words were drowned by a long, loud belch from the Dursleys’ son, Dudley. “I want more bacon.” “There’s more in the frying pan, sweetums,” said Aunt Petunia, turning misty eyes on her massive so

### Generate Embeddings

The e5 embedding model uses different prefixes for passages and queries

Therefore:

- Page content is encoded using -> passage:
- User queries will later be encoded using -> query:

Embeddings are normalized before being stored and searched

In [89]:
texts = [
    f"passage: {page['content']}"
    for page in pages
]

model = SentenceTransformer(MODEL_NAME, device=device)

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

print("Embeddings created successfully")
print("Number of embeddings:", len(embeddings))
print("Vector size:", len(embeddings[0]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Embeddings created successfully
Number of embeddings: 3568
Vector size: 1024


## 5. Qdrant Vector Database

The generated embeddings will be stored in Qdrant

Each vector will be associated with a payload containing:

- Book name
- Page number
- Page content

Cosine similarity will be used to compare the query embedding with the stored page embeddings

In [90]:
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

collection_name = QDRANT_COLLECTION

print("Connected to Qdrant")

Connected to Qdrant


In [91]:
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )


In [92]:
points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding)
    in enumerate(zip(pages, embeddings))
]

print("Points prepared:", len(points))

Points prepared: 3568


In [93]:
batch_size = 100

for start in range(0, len(points), batch_size):

    batch = points[start:start + batch_size]

    client.upsert(
        collection_name=collection_name,
        points=batch,
    )

print(f"Uploaded {len(points)} pages to Qdrant")

Uploaded 3568 pages to Qdrant


## 6. Query Router

Before searching the vector database, a Groq model classifies the user's message into one of three routes:

- retrieve: a question related to the Harry Potter books
- chitchat: greetings, thanks, or casual conversation
- off-topic: questions unrelated to the Harry Potter books

Only ( retrieve ) queries will continue to the retrieval and generation stages

In [94]:
query = input("Ask a question: ")

router_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
)

SYSTEM_PROMPT = """
You are the query router for a Harry Potter books chatbot.

Your task is to classify the user's message into exactly one of these categories:

retrieve - Use this category when the user is asking for information about the Harry Potter books, including characters, events, locations, objects, creatures, relationships, or story details

chitchat - Use this category for greetings, thank-you messages, or casual conversation that does not require searching the books

off-topic - Use this category when the message is unrelated to the Harry Potter books

Return only the category name:
retrieve
chitchat
or
off-topic

Do not provide explanations or any additional text"""

router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]

route = router_llm.invoke(router_messages).content.strip().lower()

route = route.splitlines()[0].strip(" `.,:")

if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)


Ask a question: How did Harry get the scar on his forehead?
Route: retrieve


## 7. Retrieval route

For queries classified as ( retrieve ), the query is converted into an embedding using the same embedding model

The query uses the ( query: ) prefix required by the e5 model

Qdrant then compares the query vector with the stored page vectors and returns the most similar pages

In [95]:
if route == "retrieve":

    query_vector = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=TOP_K,
    ).points

    context = ""

    for result in results:

        page = result.payload

        context += (
            f"Book: {page['book_name']}\n"
            f"Page: {page['page_number']}\n"
            f"Content: {page['content']}\n\n"
        )

        print("Score:", result.score)
        print("Book:", page["book_name"])
        print("Page:", page["page_number"])
        print("Content:", page["content"])
        print("-" * 80)

else:

    print("No Harry potter database search needed")

Score: 0.84827286
Book: Harry Potter and the Goblet of Fire
Page: 961
Content: H CHAPTER TWO THE SCAR arry lay flat on his back, breathing hard as though he had been running. He had awoken from a vivid dream with his hands pressed over his face. The old scar on his forehead, which was shaped like a bolt of lightning, was burning beneath his fingers as though someone had just pressed a white- hot wire to his skin. He sat up, one hand still on his scar, the other reaching out in the darkness for his glasses, which were on the bedside table. He put them on and his bedroom came into clearer focus, lit by a faint, misty orange light that was filtering through the curtains from the street lamp outside the window. Harry ran his fingers over the scar again. It was still painful. He turned on the lamp beside him, scrambled out of bed, crossed the room, opened his wardrobe, and peered into the mirror on the inside of the door. A skinny boy of fourteen looked back at him, his bright green eyes pu

## 8. Generation using LLM

The next step is answer generation

For ( retrieve ) queries, Gemini receives:

1. The retrieved context from Qdrant
2. The original user question

The model is instructed to answer only from the provided pages and to say that it does not know if the answer is not present in the retrieved context

In [96]:
if route == "retrieve":

    gemini_llm = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        api_key=GEMINI_API_KEY,
        temperature=0,
    )

    messages = [
        SystemMessage(
            content=(
                "Answer only from the provided pages from the Harry Potter books. "
                "If the provided context does not contain enough information to answer the question, say exactly I do not know. "
                "Do not make up or assume information that is not present in the context."
                "Keep the answer concise."
            )
        ),
        HumanMessage(
            content=f"Context:\n{context}\nQuestion:\n{query}"
        ),
    ]

    response = gemini_llm.invoke(messages)

    print("\nAnswer:")
    print(response.text)

else:

    print("no Generation needed")


Answer:
Harry got the scar when Voldemort arrived at his house, killed his parents, and turned his wand on Harry; the curse rebounded upon Voldemort, leaving Harry with the lightning-shaped cut.


## 9. Another Retrieval Test


In [97]:
test_query = "Who rescued Harry from his locked bedroom using a flying car?"

query_vector = model.encode(
    [f"query: {test_query}"],
    normalize_embeddings=True,
)[0].tolist()

search_results = client.query_points(
    collection_name=QDRANT_COLLECTION,
    query=query_vector,
    limit=TOP_K,
    with_payload=True,
).points

print("Query:", test_query)
print("=" * 80)

for result in search_results:
    page = result.payload

    print("Score:", result.score)
    print("Book:", page["book_name"])
    print("Page:", page["page_number"])
    print("Content:", page["content"])
    print("-" * 80)

Query: Who rescued Harry from his locked bedroom using a flying car?
Score: 0.81645346
Book: Harry Potter and the Goblet of Fire
Page: 967
Content: escaped before they could take him to the Ministry of Magic, and Sirius had had to flee for his life. Harry had helped him escape on the back of a hippogriff called Buckbeak, and since then, Sirius had been on the run. The home Harry might have had if Wormtail had not escaped had been haunting him all summer. It had been doubly hard to return to the Dursleys knowing that he had so nearly escaped them forever. Nevertheless, Sirius had been of some help to Harry, even if he couldn’t be with him. It was due to Sirius that Harry now had all his school things in his bedroom with him. The Dursleys had never allowed this before; their general wish of keeping Harry as miserable as possible, coupled with their fear of his powers, had led them to lock his school trunk in the cupboard under the stairs every summer prior to this. But their attitude had

## 10. Retrieval Evaluation: Precision and Recall

evaluate the retrieval stage using Precision and Recall

For each test query, define a small ground truth containing the pages that are known to be relevant to the question

- Precision measures how many of the retrieved pages are relevant
- Recall measures how many of the relevant pages were successfully retrieved

This evaluation focuses only on the retrieval component and does not involve query routing or answer generation.

In [98]:
evaluation_cases = [
    {
        "query": "Who rescued Harry from his locked bedroom using a flying car?",
        "relevant_pages": {301, 302},
    },
    {
        "query": "What loophole did Mr Weasley write into the law about enchanting a car?",
        "relevant_pages": {314},
    },
]


In [99]:
precision_scores = []
recall_scores = []

retrieved_for_evaluation = []

for case in evaluation_cases:

    query_vector = model.encode(
        [f"query: {case['query']}"],
        normalize_embeddings=True,
    )[0].tolist()

    search_results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=TOP_K,
        with_payload=True,
    ).points

    retrieved_pages = {
        result.payload["page_number"]
        for result in search_results
    }

    relevant_pages = case["relevant_pages"]

    relevant_retrieved = retrieved_pages & relevant_pages

    precision = (
        len(relevant_retrieved) / len(retrieved_pages)
        if retrieved_pages
        else 0
    )

    recall = (
        len(relevant_retrieved) / len(relevant_pages)
        if relevant_pages
        else 0
    )

    precision_scores.append(precision)
    recall_scores.append(recall)

    retrieved_for_evaluation.append(
        (case, search_results)
    )

    print("Query:", case["query"])
    print("Expected pages:", relevant_pages)
    print("Retrieved pages:", retrieved_pages)
    print("Relevant retrieved:", relevant_retrieved)
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print("-" * 60)

Query: Who rescued Harry from his locked bedroom using a flying car?
Expected pages: {301, 302}
Retrieved pages: {303, 302, 967}
Relevant retrieved: {302}
Precision: 0.33
Recall:    0.50
------------------------------------------------------------
Query: What loophole did Mr Weasley write into the law about enchanting a car?
Expected pages: {314}
Retrieved pages: {2056, 314, 467}
Relevant retrieved: {314}
Precision: 0.33
Recall:    1.00
------------------------------------------------------------


In [100]:
average_precision = sum(precision_scores) / len(precision_scores)
average_recall = sum(recall_scores) / len(recall_scores)

print(f"Average Precision: {average_precision:.2f}")
print(f"Average Recall:    {average_recall:.2f}")

Average Precision: 0.33
Average Recall:    0.75


## 11. LLM as a Judge Evaluation

In [101]:
for case, search_results in retrieved_for_evaluation:
    context = "\n\n".join(
        f"Page {result.payload['page_number']}: {result.payload['content']}"
        for result in search_results
    )

    answer = gemini_llm.invoke([
        SystemMessage(content="Answer only from the provided context. If the answer is not there, say you do not know."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}"),
    ]).text

    judge = gemini_llm.invoke([
        SystemMessage(content="""You are an evaluator for a question-answering system.
                                Judge the answer using only the context.
                                Return exactly this format:
                                Score: X/5
                                Grounded: yes or no
                                Reason: one short sentence"""),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}\n\nAnswer:\n{answer}"),
    ]).text

    print("Question:", case["query"])
    print("Answer:", answer)
    print("Judge:", judge)
    print("-" * 60)


Question: Who rescued Harry from his locked bedroom using a flying car?
Answer: Fred, George, and Ron rescued Harry.
Judge: Score: 5/5
Grounded: yes
Reason: The context explicitly describes Fred, George, and Ron arriving in a car to rescue Harry from his room.
------------------------------------------------------------
Question: What loophole did Mr Weasley write into the law about enchanting a car?
Answer: As long as the person was not intending to fly the car, the fact that the car could fly would not be against the law.
Judge: Score: 5/5
Grounded: yes
Reason: The answer accurately reflects the loophole described by Mr. Weasley in the provided text.
------------------------------------------------------------


## 12. Keyword Search Test using predefined word query

In addition to semantic retrieval using embeddings, we test the simple keyword search approach

The keyword search looks for exact query terms in the page content

This is a simple retrieval test and does not involve an LLM

In [102]:
keyword_query = "Hogwarts"

keywords = keyword_query.lower().split()

keyword_results = []

for page in pages:

    content = page["content"].lower()

    score = sum(
        content.count(keyword)
        for keyword in keywords
    )

    if score > 0:

        keyword_results.append({
            "score": score,
            "book_name": page["book_name"],
            "page_number": page["page_number"],
            "content": page["content"],
        })

keyword_results.sort(
    key=lambda result: result["score"],
    reverse=True
)

print("Keyword Query:", keyword_query)
print("=" * 80)

for result in keyword_results[:TOP_K]:

    print("Keyword score:", result["score"])
    print("Book:", result["book_name"])
    print("Page:", result["page_number"])
    print("Content:", result["content"])
    print("-" * 80)

Keyword Query: Hogwarts
Keyword score: 5
Book: Harry Potter and the Order of the Phoenix
Page: 1865
Content: “This is not the first time in recent weeks Fudge has used new laws to effect improvements at the Wizarding school. As recently as August 30th Educational Decree Twenty-two was passed, to ensure that, in the event of the current headmaster being unable to provide a candidate for a teaching post, the Ministry should select an appropriate person. “‘That’s how Dolores Umbridge came to be appointed to the teaching staff at Hogwarts,’ said Weasley last night. ‘Dumbledore couldn’t find anyone, so the Minister put in Umbridge and of course, she’s been an immediate success —’” “She’s been a WHAT?” said Harry loudly. “Wait, there’s more,” said Hermione grimly. “‘— an immediate success, totally revolutionizing the teaching of Defense Against the Dark Arts and providing the Minister with on-the-ground feedback about what’s really happening at Hogwarts.’ “It is this last function that the M

## 13. Another Retrieval Test with Multiple Queries

To further verify the retrieval system, will test it with 10 questions covering different characters, events, and books

For each query the system retrieves the top relevant pages

In [103]:
test_queries = [
    "Who is Harry Potter's godfather?",
    "Who gave Harry the Marauder's Map?",
    "What is the name of Hermione's cat?",
    "Who killed Dumbledore?",
    "What happened to Cedric Diggory?",
    "What is the name of Ron Weasley's pet rat?",
    "What is the name of Hagrid's giant dog?",
    "Who is the headmaster of Hogwarts at the beginning of the series?",
    "What object determines a student's Hogwarts house?",
    "Who was Harry Potter's first Defense Against the Dark Arts teacher?"
]

for query in test_queries:

    query_vector = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=TOP_K,
        with_payload=True,
    ).points

    print("\nQuery:", query)
    print("-" * 80)

    for result in results:

        page = result.payload

        print(
            f"Book: {page['book_name']} | "
            f"Page: {page['page_number']} | "
            f"Score: {result.score:.4f}"
        )

    print("=" * 80)


Query: Who is Harry Potter's godfather?
--------------------------------------------------------------------------------
Book: Harry Potter and the Goblet of Fire | Page: 966 | Score: 0.8184
Book: Harry Potter and the Deathly Hallows | Page: 2987 | Score: 0.8173
Book: Harry Potter and the Prisoner of Azkaban | Page: 939 | Score: 0.8166

Query: Who gave Harry the Marauder's Map?
--------------------------------------------------------------------------------
Book: Harry Potter and the Goblet of Fire | Page: 1346 | Score: 0.8435
Book: Harry Potter and the Prisoner of Azkaban | Page: 813 | Score: 0.8428
Book: Harry Potter and the Goblet of Fire | Page: 1344 | Score: 0.8330

Query: What is the name of Hermione's cat?
--------------------------------------------------------------------------------
Book: Harry Potter and the Prisoner of Azkaban | Page: 622 | Score: 0.8205
Book: Harry Potter and the Half-Blood Prince | Page: 2526 | Score: 0.8185
Book: Harry Potter and the Prisoner of Azkaban